# 07B 一次成功和稳定成功

一个任务做四次，成功两次。我们想问两个不同问题：给两次机会，至少一次成功的可能性有多大？两次都成功的可能性又有多大？

先用枚举回答，再看公式。以下布尔序列是手工教学样例，真实结果在后半部分读取。

本实践还会把 BFCL、GAIA、SWE-bench 与 τ-bench 分别缩小成一条课程级测例，明确任务输入、可观察证据和本课对应能力；这是测量思想映射，不宣称复现官方排行榜。

**贯穿案例**

沿用四道仓库阅读任务，先用可手算的成功序列区分“两次中至少一次成功”和“两次都成功”，再比较任务等权平均与把试验混在一起的结果。`pass_metrics` 分别计算这两种组合问题；这一步是教学计算，不是给四题真实试验重新造一个官方基准分数。指标说明书保留样本与计算口径。


## 学习路线与配套课件

本 Notebook 可以独立从头运行。先读定义和输入，预测结果，再执行并解释差异。小算例帮助理解机制；真实训练、工具执行与保存的 API 记录会分别标明来源。

本节重点：增加尝试次数，两个指标为什么向相反方向变。

课件页：L07.04-S01、L07.04-S02。

沿原有课程单元顺序完成实验；新增对照放在相关代码后。练习先自行作答，展开参考分析后再比较。

In [1]:
from pathlib import Path
import sys, json, copy, math, statistics, hashlib
from collections import defaultdict
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "00-资料库使用说明.md").exists())
CHAPTER = ROOT / "07 智能体评估与对照实验"
sys.path.insert(0, str(CHAPTER / "代码"))
from evaluation_lab import parse_answer, typed_equal, observed_files, grade_trial, pass_metrics, summarize, diagnose_format
DATASET = json.loads((CHAPTER / "数据/tasks.json").read_text(encoding='utf-8'))
TASKS = {task["id"]:task for task in DATASET["tasks"]}
print("四项开发集任务；真实请求默认关闭。")


四项开发集任务；真实请求默认关闭。


## 1. 结果和过程分别记录

结果指标可以是任务通过率、事实与引用检查；过程指标可以是模型请求数、工具调用数、耗时和已知 Token 用量。

多次工具调用仍可能属于同一试验。调用少不自动代表好，输出长也不自动代表好；要和完成质量一起看。请求异常时本地未取得 usage，不能把缺失用量当成零。


In [2]:
record = json.loads((CHAPTER / "数据/实验记录/deepseek-evaluation.json").read_text(encoding='utf-8'))
trial = record["trials"][0]
print("任务 / 试验 / 版本:", trial["task_id"], trial["repeat"], trial["variant"])
print("状态:", trial["state"]["status"], "模型请求:", trial["state"]["turns"], "工具调用:", trial["state"]["tool_calls"])
print("耗时秒:", trial["elapsed_seconds"], "规则检查:", trial["grade"]["checks"])
print("usage 返回条数:", len(trial.get("observed_request_usage", trial["state"]["usage"])))


任务 / 试验 / 版本: relative-base 1 base
状态: final_answer 模型请求: 3 工具调用: 6
耗时秒: 17.9644 规则检查: {'completed': True, 'schema': True, 'facts': True, 'citations': True, 'source_unchanged': True}
usage 返回条数: 3


## 2. 从四次里选两次

试验结果为 [成功、失败、成功、失败]。从四次中等可能选两个位置，一共六种组合；至少一个成功的有五种，全部成功的有一种。

这里的“抽两个已记录试验”是解释估计量，不是在运行 Agent 时从失败轨迹中挑最好看的步骤。


In [3]:
from itertools import combinations
outcomes = [True, False, True, False]
pairs = list(combinations(range(len(outcomes)), 2))
for pair in pairs:
    selected = [outcomes[i] for i in pair]
    print(pair, selected, "至少一次:", any(selected), "全部:", all(selected))
print("至少一次:", sum(any(outcomes[i] for i in p) for p in pairs) / len(pairs))
print("全部成功:", sum(all(outcomes[i] for i in p) for p in pairs) / len(pairs))


(0, 1) [True, False] 至少一次: True 全部: False
(0, 2) [True, True] 至少一次: True 全部: True
(0, 3) [True, False] 至少一次: True 全部: False
(1, 2) [False, True] 至少一次: True 全部: False
(1, 3) [False, False] 至少一次: False 全部: False
(2, 3) [True, False] 至少一次: True 全部: False
至少一次: 0.8333333333333334
全部成功: 0.16666666666666666


## 3. pass@k 与 pass^k

同一任务独立试验 n 次，成功 c 次，k 不大于 n。按组合计数：

$$\widehat{\mathrm{pass@k}}=1-\frac{\binom{n-c}{k}}{\binom{n}{k}},\qquad
\widehat{\mathrm{pass^k}}=\frac{\binom{c}{k}}{\binom{n}{k}}.$$

前者关心至少一次成功，后者关心全部成功；多个任务先各算一次，再等权平均。上例得到 5/6 和 1/6。

若已知某任务每次成功概率 p，独立时理论概率是 1-(1-p)^k 与 p^k。把小样本 c/n 直接代入会得到另一种估计，不能误称就是上述组合估计。


In [4]:
def pass_metrics(outcomes, k):
    """一个任务 n 次独立试验中抽取 k 次的无偏估计；不同任务最后等权平均。"""
    if not outcomes or any(type(x) is not bool for x in outcomes): raise ValueError('结果必须是非空布尔列表')
    n=len(outcomes);c=sum(outcomes)
    if type(k) is not int or not 1<=k<=n: raise ValueError('需要 1 <= k <= n')
    combinations=math.comb(n,k)
    return {'pass_at_k':1-(math.comb(n-c,k) if n-c>=k else 0)/combinations,
            'pass_power_k':(math.comb(c,k) if c>=k else 0)/combinations}


In [5]:
print("组合估计:", pass_metrics(outcomes, 2))
p = sum(outcomes) / len(outcomes)
print("把样本比例直接代入:", {"at_least_one":1-(1-p)**2, "all":p**2})
assert math.isclose(pass_metrics(outcomes, 2)["pass_at_k"], 5/6)
assert math.isclose(pass_metrics(outcomes, 2)["pass_power_k"], 1/6)
# 0.75 与 0.25 不等于 5/6 与 1/6；先说明自己计算的是哪一个量。


组合估计: {'pass_at_k': 0.8333333333333334, 'pass_power_k': 0.16666666666666666}
把样本比例直接代入: {'at_least_one': 0.75, 'all': 0.25}


### 增加尝试次数，两个指标为什么向相反方向变

配套课件：L07.04-S01、L07.04-S02。

固定四次观察结果 `[成功, 失败, 成功, 失败]`，从中不放回选 k 次。pass@k 问至少一次成功，pass^k 问选中的全部成功。本课组合估计分别为 $1-\binom{n-c}{k}/\binom{n}{k}$ 和 $\binom{c}{k}/\binom{n}{k}$。

In [6]:
case_outcomes = [True, False, True, False]
for case_k in range(1, 5):
    case_metric = pass_metrics(case_outcomes, case_k)
    case_groups = list(combinations(case_outcomes, case_k))
    case_at_least = sum(any(g) for g in case_groups) / len(case_groups)
    case_all = sum(all(g) for g in case_groups) / len(case_groups)
    assert math.isclose(case_metric["pass_at_k"], case_at_least)
    assert math.isclose(case_metric["pass_power_k"], case_all)
    print("k =", case_k, "组合数:", len(case_groups), "至少一次:", round(case_at_least, 4),
          "全部成功:", round(case_all, 4))


k = 1 组合数: 4 至少一次: 0.5 全部成功: 0.5
k = 2 组合数: 6 至少一次: 0.8333 全部成功: 0.1667
k = 3 组合数: 4 至少一次: 1.0 全部成功: 0.0
k = 4 组合数: 1 至少一次: 1.0 全部成功: 0.0


**结果解读**

k=2 时为 5/6 和 1/6；k=4 时至少一次为 1，全部成功为 0。更高 pass@k 不能直接说明单次更可靠，也不等于最终用户有能力挑出正确答案。

**小练习**

如果每次失败都会造成不可撤销的后果，应特别关注哪个问题？为什么不能只报 pass@k？

<details><summary>完成后展开参考分析</summary>

需关注持续/全次成功及失败后果；至少一次成功会掩盖其余失败。指标要对应真实使用方式和错误成本。

</details>

公式与实现来源见 [配套素材来源](../../../references/companion-upgrade-sources.md)。

## 4. 先按任务计算，再平均

任务 A 运行一次成功，任务 B 运行三次都失败。任务等权的 pass@1 是 (1+0)/2=0.5；把所有试验池化得到 1/4=0.25，会让重复次数多的任务占更大权重。

本次真实实验计划每题次数相同；若中途停止，则先说明数据不完整，不偷偷把不平衡试验当成同一套完整对照。


In [7]:
groups = [[True], [False,False,False]]
macro = statistics.mean(pass_metrics(group,1)["pass_at_k"] for group in groups)
pooled = sum(sum(group) for group in groups) / sum(map(len,groups))
print("任务等权:", macro, "试验池化:", pooled)


任务等权: 0.5 试验池化: 0.25


## 5. 基准测量的任务不同

| 基准 | 一条直观测例 | 主要结果证据 | 课程中的轻量映射 |
|---|---|---|---|
| BFCL | 给定函数定义与用户请求，选择正确工具并生成合法参数 | 函数名、参数结构和值 | 固定工具 Schema，检查调用选择与参数 |
| GAIA | 回答一个需要检索和多步推理、但具有唯一短答案的问题 | 最终答案与证据使用 | 固定资料，检查答案和可回查证据链 |
| SWE-bench | 根据真实仓库 issue 生成补丁并在仓库环境运行测试 | 代码补丁、FAIL_TO_PASS 与 PASS_TO_PASS 测试 | 对同一源码改动运行目标测试和回归测试 |
| τ-bench | 在业务策略约束下调用工具并与用户交互 | 最终数据库状态、必要回复和过程约束 | 同时检查任务结果、环境状态与策略边界 |

不能把不同任务、版本、工具与预算下的分数拼成一个通用能力结论。τ-bench 原论文也指出，最终状态通过仍可能漏掉中途未取得确认等违规行为。

本课不要求下载完整数据或生成官方分数。这里先把每个基准拆成“任务单元—环境—证据—评分边界”，再说明它与我们的四题导读评估有什么不同。


In [8]:
benchmarks = [
    {"name":"BFCL", "task_unit":"函数调用请求", "evidence":"函数名、参数结构和值", "course_case":"固定 Schema 的工具选择与参数检查", "source":"https://gorilla.cs.berkeley.edu/blogs/8_berkeley_function_calling_leaderboard.html"},
    {"name":"GAIA", "task_unit":"有唯一短答案的通用助手问题", "evidence":"最终答案与证据使用", "course_case":"固定资料的答案与证据链核验", "source":"https://arxiv.org/abs/2311.12983"},
    {"name":"SWE-bench", "task_unit":"真实仓库 issue", "evidence":"补丁与两组测试", "course_case":"目标测试加回归测试", "source":"https://www.swebench.com/"},
    {"name":"tau-bench", "task_unit":"策略约束下的工具与用户交互", "evidence":"最终状态、回复与过程约束", "course_case":"结果、状态和策略边界联合检查", "source":"https://arxiv.org/html/2406.12045v1"},
]
for row in benchmarks:
    print(f"{row['name']}: {row['task_unit']} -> {row['evidence']} -> {row['course_case']}")
assert {row["name"] for row in benchmarks} == {"BFCL", "GAIA", "SWE-bench", "tau-bench"}
# 这里是测量结构映射，不下载数据、不运行官方环境、不生成榜单分数。


BFCL: 函数调用请求 -> 函数名、参数结构和值 -> 固定 Schema 的工具选择与参数检查
GAIA: 有唯一短答案的通用助手问题 -> 最终答案与证据使用 -> 固定资料的答案与证据链核验
SWE-bench: 真实仓库 issue -> 补丁与两组测试 -> 目标测试加回归测试
tau-bench: 策略约束下的工具与用户交互 -> 最终状态、回复与过程约束 -> 结果、状态和策略边界联合检查


## 练习：修改成功序列

把四次结果改成三次成功、一次失败，先手算 k=2，再验证。进一步解释：两次真实试验都成功，能否证明未来始终成功？


In [9]:
changed = [True, True, True, False]
print(pass_metrics(changed, 2))


{'pass_at_k': 1.0, 'pass_power_k': 0.5}


In [10]:
assert pass_metrics(changed, 2) == {"pass_at_k":1.0, "pass_power_k":0.5}
print("两个样本全部成功只是观测结果。仍需要更多试验、新任务及稳定设置来了解可靠性。")


两个样本全部成功只是观测结果。仍需要更多试验、新任务及稳定设置来了解可靠性。


## 可带走的结果：可靠性指标说明书

把教学样例的枚举结果、组合估计、任务等权汇总和基准适用边界写入 JSON。它不仅给出数字，也标明数字回答的是“至少一次成功”还是“每次都成功”，以及哪些条件不能从小样本推出。


In [11]:
report_dir = ROOT / ".local/student-runs/l07"
report_dir.mkdir(parents=True, exist_ok=True)
report = {
    "report_type": "reliability-metrics-note",
    "run_model": record["model"],
    "model_selection_note": "Before a live rerun, list models available to the current account and choose a supported one.",
    "teaching_sequence": outcomes,
    "k2_combination_estimate": pass_metrics(outcomes, 2),
    "k2_plugin_estimate": {"pass_at_k": 1 - (1 - p) ** 2, "pass_power_k": p ** 2},
    "changed_sequence": changed,
    "changed_k2": pass_metrics(changed, 2),
    "aggregation_example": {"task_macro_pass_at_1": macro, "pooled_trial_rate": pooled},
    "benchmark_scope": benchmarks,
    "interpretation": {
        "pass_at_k": "chance of at least one success among k independent attempts",
        "pass_power_k": "chance that all k independent attempts succeed",
        "limits": ["small samples do not prove future reliability", "task and environment definitions travel with every score"],
    },
}
report_path = report_dir / "reliability-metrics-note.json"
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print("已保存可靠性指标说明书:", report_path.relative_to(ROOT))
print(json.dumps(report, ensure_ascii=False, indent=2))


已保存可靠性指标说明书: .local\student-runs\l07\reliability-metrics-note.json
{
  "report_type": "reliability-metrics-note",
  "run_model": "deepseek-flash",
  "model_selection_note": "Before a live rerun, list models available to the current account and choose a supported one.",
  "teaching_sequence": [
    true,
    false,
    true,
    false
  ],
  "k2_combination_estimate": {
    "pass_at_k": 0.8333333333333334,
    "pass_power_k": 0.16666666666666666
  },
  "k2_plugin_estimate": {
    "pass_at_k": 0.75,
    "pass_power_k": 0.25
  },
  "changed_sequence": [
    true,
    true,
    true,
    false
  ],
  "changed_k2": {
    "pass_at_k": 1.0,
    "pass_power_k": 0.5
  },
  "aggregation_example": {
    "task_macro_pass_at_1": 0.5,
    "pooled_trial_rate": 0.25
  },
  "benchmark_scope": [
    {
      "name": "BFCL",
      "task_unit": "函数调用请求",
      "evidence": "函数名、参数结构和值",
      "course_case": "固定 Schema 的工具选择与参数检查",
      "source": "https://gorilla.cs.berkeley.edu/blogs/8_berkeley_function_ca

## 回顾

先确定成功事件，再选择统计量；按任务统计，失败留在分母；资源与质量一起分析；基准分数必须带任务与环境条件。

公式来源：[τ-bench §3](https://arxiv.org/html/2406.12045v1)。
